<a href="https://colab.research.google.com/github/silvishVictmori/Apache-Spark-Setup/blob/main/pandas3_2_db.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Load the Drive helper and mount
from google.colab import drive

# This will prompt for authorization.
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# remove old spark installations if needed
!rm -dr spark*

In [ ]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
# !wget -q https://downloads.apache.org/spark/spark-3.2.0/spark-3.2.0-bin-hadoop2.7.tgz
!wget -q https://downloads.apache.org/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
# !tar xf spark-3.2.1-bin-hadoop2.7.tgz
!tar xf spark-3.2.1-bin-hadoop3.2.tgz



In [ ]:
!ls /content/

drive		       sample_data		  spark-3.2.1-bin-hadoop3.2.tgz
postgresql-42.3.2.jar  spark-3.2.1-bin-hadoop3.2


In [ ]:
!pip install -q findspark
!pip install pyspark

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
# os.environ["SPARK_HOME"] = "/content/spark-3.2-bin-hadoop3.2"
os.environ["SPARK_HOME"] = "/content/spark-3.2.1-bin-hadoop3.2"

# !update-alternatives --set java /usr/lib/jvm/java-8-openjdk-amd64/jre/bin/java

!java -version
!python --version

openjdk version "11.0.13" 2021-10-19
OpenJDK Runtime Environment (build 11.0.13+8-Ubuntu-0ubuntu1.18.04)
OpenJDK 64-Bit Server VM (build 11.0.13+8-Ubuntu-0ubuntu1.18.04, mixed mode, sharing)
Python 3.7.12


In [ ]:
import findspark
#findspark.init("/content/spark-2.4.4-bin-hadoop2.7")
findspark.init()

In [ ]:
!wget https://jdbc.postgresql.org/download/postgresql-42.3.2.jar

import os
from pprint import pprint
pprint(os.listdir('/content'))

--2022-02-20 22:51:42--  https://jdbc.postgresql.org/download/postgresql-42.3.2.jar
Resolving jdbc.postgresql.org (jdbc.postgresql.org)... 72.32.157.228, 2001:4800:3e1:1::228
Connecting to jdbc.postgresql.org (jdbc.postgresql.org)|72.32.157.228|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1040162 (1016K) [application/java-archive]
Saving to: ‘postgresql-42.3.2.jar.1’

postgresql-42.3.2.j 100%[===================>]   1016K  --.-KB/s    in 0.1s    

2022-02-20 22:51:42 (6.71 MB/s) - ‘postgresql-42.3.2.jar.1’ saved [1040162/1040162]

['.config',
 'postgresql-42.3.2.jar',
 'spark-3.2.1-bin-hadoop3.2',
 'postgresql-42.3.2.jar.1',
 'spark-3.2.1-bin-hadoop3.2.tgz',
 'drive',
 'sample_data']


In [ ]:
os.environ["SPARK_HOME"]

'/content/spark-3.2.1-bin-hadoop3.2'

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('SparkWithPostgres')\
        .config("spark.driver.extraClassPath", "/content/postgresql-42.3.2.jar")\
        .getOrCreate()
spark

In [ ]:
# import Pandas-on-Spark
import pyspark.pandas as ps
# Create a DataFrame with Pandas-on-Spark
ps_df = ps.DataFrame(range(10))
# Convert a Pandas-on-Spark Dataframe into a Pandas Dataframe
pd_df = ps_df.to_pandas()
# Convert a Pandas Dataframe into a Pandas-on-Spark Dataframe
ps_df = ps.from_pandas(pd_df)

In [ ]:
ps_df

,0
0,0
1,1
2,2
3,3
4,4
5,5
6,6
7,7
8,8
9,9


In [ ]:
postgres_uri = "jdbc:postgresql://depgdb.crhso94tou3n.eu-west-2.rds.amazonaws.com:5432/addyourdbUSERNAME"
dbtable = "information_schema.schemata"
user = "addyourdbUSERNAME"
password = "addyourPASWORD"

In [ ]:
df = spark.read \
    .format("jdbc") \
    .option("url", postgres_uri) \
    .option("dbtable", dbtable) \
    .option("user", user) \
    .option("password", password) \
    .option("driver", "org.postgresql.Driver") \
    .load()

df.printSchema()

root
 |-- catalog_name: string (nullable = true)
 |-- schema_name: string (nullable = true)
 |-- schema_owner: string (nullable = true)
 |-- default_character_set_catalog: string (nullable = true)
 |-- default_character_set_schema: string (nullable = true)
 |-- default_character_set_name: string (nullable = true)
 |-- sql_path: string (nullable = true)



In [ ]:
df_users = spark.read \
    .format("jdbc") \
    .option("url", postgres_uri) \
    .option("user", user) \
    .option("password", password) \
    .option("driver", "org.postgresql.Driver") \
    .option("query", "select * from ucl_messenger.users") \
    .load()

df_users.count()

10